In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/riverload_starter.ipynb
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/sample_submission.csv
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/README.md
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/MANIFEST.sha256
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/to_kaggle.py
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/history/levels_hourly.parquet
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/history/discharge_hourly.parquet
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/riverload/data.py
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/riverload/itinerary.py
/kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data/riverload/q

In [25]:
import glob, os, sys, time

In [26]:
import glob, os, sys, time
import pandas as pd

CHALLENGE = None

def kind(d):
    return 'C1' if 'dmin_q50_cm' in pd.read_csv(
        f'{d}/sample_submission.csv', nrows=1
    ).columns else 'C2'

cands = sorted({
    os.path.dirname(os.path.dirname(p))
    for p in glob.glob(
        '/kaggle/input/**/reference/route_pieces.csv',
        recursive=True
    )
}) or ['.']

if CHALLENGE:
    cands = [d for d in cands if kind(d) == CHALLENGE] or cands
elif len({kind(d) for d in cands}) > 1:
    raise ValueError(
        'Both competitions are attached: set CHALLENGE = "C1" or "C2" above'
    )

DATA = cands[0]
CHALLENGE = kind(DATA)

sys.path.insert(0, DATA)

from riverload import Env
from riverload.data import Data
from riverload.quickstart import persistence_forecast, decide
from to_kaggle import c1_to_kaggle, c2_to_kaggle

sample = pd.read_csv(f'{DATA}/sample_submission.csv')

print('data:', DATA, '| challenge:', CHALLENGE)

data: /kaggle/input/competitions/riverload-c1-forecast-the-rhine-test1/riverload-data | challenge: C1


In [27]:
test = Env(DATA, 'test')
sc = test.scenarios.iloc[0]

print(sc.to_string())

tt = test.timetable(sc.scenario_id)

tt[
    (tt.route_id == tt.route_id.iloc[0]) &
    (tt.departure_slot == 12)
].head(12)

scenario_id               TS-OOKXQ1
issue_id                   I-4XV8PL
family                            A
origin                    Rotterdam
ports                      Duisburg
routes                        A1;A2
issue_time      2022-02-01 06:00:00
t0              2022-02-06 06:00:00
n_containers                    239


,route_id,departure_slot,leg,order,kind,segment_id,equiv_depth_cm,t_enter,t_exit,lock_h,scenario_id
84,A1,12,1,0,TIDAL,None,470.0,2022-02-06 06:00:00.000000000,2022-02-06 08:59:30.229007633,0.0,TS-OOKXQ1
85,A1,12,1,1,RIVER,D02b,NaN,2022-02-06 08:59:30.229007633,2022-02-06 11:41:54.415054144,0.0,TS-OOKXQ1
86,A1,12,1,2,RIVER,D02a,NaN,2022-02-06 11:41:54.415054144,2022-02-06 14:27:47.903426237,0.0,TS-OOKXQ1
87,A1,12,1,3,RIVER,D01,NaN,2022-02-06 14:27:47.903426237,2022-02-06 15:04:37.205751818,0.0,TS-OOKXQ1
88,A1,12,1,4,RIVER,S17,NaN,2022-02-06 15:04:37.205751818,2022-02-06 16:19:37.205751818,0.0,TS-OOKXQ1
89,A1,12,1,5,RIVER,S16,NaN,2022-02-06 16:19:37.205751818,2022-02-06 19:00:52.205751818,0.0,TS-OOKXQ1
90,A1,12,1,6,RIVER,S15,NaN,2022-02-06 19:00:52.205751818,2022-02-06 19:50:22.205751818,0.0,TS-OOKXQ1


In [28]:
train = Env(DATA, 'train')

In [29]:

# Generate predictions for the test data
fc = persistence_forecast(test, Data(DATA, 'test', env=test))

# Convert predictions into the required C1 submission format
if CHALLENGE == 'C1':
    sub = c1_to_kaggle(fc)
else:
    dec, lp = decide(test, fc, slots=[12])
    sub = c2_to_kaggle(dec, lp, test.manifests)

# Verify that IDs and columns match the official template
assert set(sub.id) == set(sample.id)
assert list(sub.columns) == list(sample.columns)

# Save the submission file
sub.to_csv('/kaggle/working/submission.csv', index=False)

print("Submission shape:", sub.shape)
print(sub.head())
print("Saved to /kaggle/working/submission.csv")

Submission shape: (9400, 5)
                 id  dmin_q05_cm  dmin_q10_cm  dmin_q25_cm  dmin_q50_cm
0  TS-OOKXQ1_A1_0_1        302.2        329.2        362.2        385.2
1  TS-OOKXQ1_A1_1_1        302.2        329.2        362.2        385.2
2  TS-OOKXQ1_A1_2_1        302.2        329.2        362.2        385.2
3  TS-OOKXQ1_A1_3_1        302.2        329.2        362.2        385.2
4  TS-OOKXQ1_A1_4_1        302.2        329.2        362.2        385.2
Saved to /kaggle/working/submission.csv


In [30]:
print("Training scenarios:", len(train.scenarios))
print("Test scenarios:", len(test.scenarios))

data_train = Data(DATA, 'train', env=train)

print("Data object created:", type(data_train))

Training scenarios: 596
Test scenarios: 94
Data object created: <class 'riverload.data.Data'>


In [31]:

# Inspect what the official data API provides
print("Training scenarios:", len(train.scenarios))
print("Test scenarios:", len(test.scenarios))

data_train = Data(DATA, 'train', env=train)

print("\nData object methods:")
print([name for name in dir(data_train) if not name.startswith("_")])

Training scenarios: 596
Test scenarios: 94

Data object methods:
['depth_from_levels', 'discharge', 'levels', 'scenarios', 'segments', 'split', 'windows']


In [32]:

# Inspect the structure of the training data
print("=== SCENARIOS ===")
print(data_train.scenarios.head())
print("\nScenario columns:")
print(data_train.scenarios.columns.tolist())

print("\n=== LEVELS ===")
levels = data_train.levels
print("Type:", type(levels))
print("Shape:", getattr(levels, "shape", "not a DataFrame"))
print(levels.head() if hasattr(levels, "head") else levels)

print("\n=== DISCHARGE ===")
discharge = data_train.discharge
print("Type:", type(discharge))
print("Shape:", getattr(discharge, "shape", "not a DataFrame"))
print(discharge.head() if hasattr(discharge, "head") else discharge)

print("\n=== WINDOWS METHOD ===")
print(data_train.windows)

=== SCENARIOS ===
  scenario_id  issue_id family     origin                        ports routes  \
0   TR-OY4HUB  I-BUMHK3      A  Rotterdam                     Duisburg  A1;A2   
1   TR-LMQZDM  I-BUMHK3      B  Rotterdam  Duisburg;Mannheim;Karlsruhe  B1;B2   
2   TR-MC096A  I-20CXLA      A  Rotterdam                     Duisburg  A1;A2   
3   TR-LKYT1T  I-20CXLA      B  Rotterdam  Duisburg;Mannheim;Karlsruhe  B1;B2   
4   TR-NJ2T7X  I-VDVPI9      A  Rotterdam                     Duisburg  A1;A2   

           issue_time                  t0  n_containers  
0 2015-01-30 06:00:00 2015-02-04 06:00:00           221  
1 2015-01-30 06:00:00 2015-02-04 06:00:00           232  
2 2015-02-04 06:00:00 2015-02-09 06:00:00           192  
3 2015-02-04 06:00:00 2015-02-09 06:00:00           184  
4 2015-02-11 06:00:00 2015-02-16 06:00:00           193  

Scenario columns:
['scenario_id', 'issue_id', 'family', 'origin', 'ports', 'routes', 'issue_time', 't0', 'n_containers']

=== LEVELS ===
Type: <cl

In [33]:

import inspect
import os

print("=== levels ===")
print("Signature:", inspect.signature(data_train.levels))
print(inspect.getdoc(data_train.levels))

print("\n=== discharge ===")
print("Signature:", inspect.signature(data_train.discharge))
print(inspect.getdoc(data_train.discharge))

print("\n=== Training data files ===")
for root, dirs, files in os.walk(DATA):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    for f in files:
        if f.endswith((".csv", ".parquet")):
            print(os.path.relpath(os.path.join(root, f), DATA))

print("\n=== Data object attributes ===")
print(vars(data_train).keys())

=== levels ===
Signature: (sid, issue, days=21)
None

=== discharge ===
Signature: (sid, issue, days=21)
None

=== Training data files ===
sample_submission.csv
history/levels_hourly.parquet
history/discharge_hourly.parquet
reference/routes.csv
reference/ship_slots.csv
reference/imputed_hours.csv
reference/discharge_series.csv
reference/segments.csv
reference/route_pieces.csv
reference/gauges.csv
test/window_levels_hourly.parquet
test/lock_queues.csv
test/manifests.csv
test/window_discharge_hourly.parquet
test/scenarios.csv
train/labels_dmin.csv
train/labels_best_value.csv
train/lock_queues.csv
train/manifests.csv
train/scenarios.csv

=== Data object attributes ===
dict_keys(['split', 'segments', 'scenarios', '_issue', '_lev', '_q', '_env', '_windows'])


In [34]:

import pandas as pd

# Load the true training labels
labels_path = os.path.join(DATA, "train", "labels_dmin.csv")
labels = pd.read_csv(labels_path)

print("=== TRAINING LABELS ===")
print("Shape:", labels.shape)
print("Columns:", labels.columns.tolist())
print(labels.head())
print(labels.dtypes)

# Select one training scenario
sc = train.scenarios.iloc[0]
sid = sc["scenario_id"]
issue = sc["issue_time"]

print("\n=== ONE SCENARIO ===")
print("Scenario ID:", sid)
print("Issue time:", issue)

# Retrieve the 21 days of historical measurements
lev = data_train.levels(sid, issue)
dis = data_train.discharge(sid, issue)

print("\n=== WATER LEVELS ===")
print("Type:", type(lev))
print("Shape:", getattr(lev, "shape", None))
print(lev.head() if hasattr(lev, "head") else lev)

print("\n=== DISCHARGE ===")
print("Type:", type(dis))
print("Shape:", getattr(dis, "shape", None))
print(dis.head() if hasattr(dis, "head") else dis)

=== TRAINING LABELS ===
Shape: (59600, 5)
Columns: ['scenario_id', 'route_id', 'departure_slot', 'leg', 'dmin_cm']
  scenario_id route_id  departure_slot  leg  dmin_cm
0   TR-OY4HUB       A1               0    1    493.4
1   TR-OY4HUB       A1               1    1    493.0
2   TR-OY4HUB       A1               2    1    504.2
3   TR-OY4HUB       A1               3    1    506.4
4   TR-OY4HUB       A1               4    1    514.7
scenario_id        object
route_id           object
departure_slot      int64
leg                 int64
dmin_cm           float64
dtype: object

=== ONE SCENARIO ===
Scenario ID: TR-OY4HUB
Issue time: 2015-01-30 06:00:00

=== WATER LEVELS ===
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (504, 21)
                             G01         G02         G03         G04  \
timestamp                                                              
2015-01-09 06:00:00  1079.800049  170.500000  303.500000  325.299988   
2015-01-09 07:00:00  1079.800049  169.500000  3

In [35]:
import pandas as pd

labels_path = os.path.join(DATA, "train", "labels_dmin.csv")
labels = pd.read_csv(labels_path)

print("=== TRAINING LABELS ===")
print("Shape:", labels.shape)
print("Columns:", labels.columns.tolist())
print(labels.head())
print(labels.dtypes)

sc = train.scenarios.iloc[0]
sid = sc["scenario_id"]
issue = sc["issue_time"]

print("\n=== ONE SCENARIO ===")
print("Scenario ID:", sid)
print("Issue time:", issue)

lev = data_train.levels(sid, issue)
dis = data_train.discharge(sid, issue)

print("\n=== WATER LEVELS ===")
print("Type:", type(lev))
print("Shape:", getattr(lev, "shape", None))
print(lev.head() if hasattr(lev, "head") else lev)

print("\n=== DISCHARGE ===")
print("Type:", type(dis))
print("Shape:", getattr(dis, "shape", None))
print(dis.head() if hasattr(dis, "head") else dis)

=== TRAINING LABELS ===
Shape: (59600, 5)
Columns: ['scenario_id', 'route_id', 'departure_slot', 'leg', 'dmin_cm']
  scenario_id route_id  departure_slot  leg  dmin_cm
0   TR-OY4HUB       A1               0    1    493.4
1   TR-OY4HUB       A1               1    1    493.0
2   TR-OY4HUB       A1               2    1    504.2
3   TR-OY4HUB       A1               3    1    506.4
4   TR-OY4HUB       A1               4    1    514.7
scenario_id        object
route_id           object
departure_slot      int64
leg                 int64
dmin_cm           float64
dtype: object

=== ONE SCENARIO ===
Scenario ID: TR-OY4HUB
Issue time: 2015-01-30 06:00:00

=== WATER LEVELS ===
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (504, 21)
                             G01         G02         G03         G04  \
timestamp                                                              
2015-01-09 06:00:00  1079.800049  170.500000  303.500000  325.299988   
2015-01-09 07:00:00  1079.800049  169.500000  3

In [36]:
import pandas as pd
import os

# =========================
# 1. TARGET / LABELS
# =========================
labels_path = os.path.join(DATA, "train", "labels_dmin.csv")
labels = pd.read_csv(labels_path)

print("========== LABELS ==========")
print("Shape:", labels.shape)
print("Columns:")
print(labels.columns.tolist())
print("\nFirst 5 rows:")
display(labels.head())
print("\nDtypes:")
print(labels.dtypes)


# =========================
# 2. ONE TRAINING SCENARIO
# =========================
sc = train.scenarios.iloc[0]

sid = sc["scenario_id"]
issue = sc["issue_time"]

print("\n========== SCENARIO ==========")
print("scenario_id:", sid)
print("issue_time:", issue)


# =========================
# 3. WATER LEVELS
# =========================
lev = data_train.levels(sid, issue)

print("\n========== WATER LEVELS ==========")
print("Type:", type(lev))
print("Shape:", getattr(lev, "shape", None))
display(lev.head())
print("\nColumns/index:")
print(lev.columns.tolist() if hasattr(lev, "columns") else lev.index)


# =========================
# 4. DISCHARGE
# =========================
dis = data_train.discharge(sid, issue)

print("\n========== DISCHARGE ==========")
print("Type:", type(dis))
print("Shape:", getattr(dis, "shape", None))
display(dis.head())
print("\nColumns/index:")
print(dis.columns.tolist() if hasattr(dis, "columns") else dis.index)

========== LABELS ==========
Shape: (59600, 5)
Columns:
['scenario_id', 'route_id', 'departure_slot', 'leg', 'dmin_cm']

First 5 rows:


,scenario_id,route_id,departure_slot,leg,dmin_cm
0,TR-OY4HUB,A1,0,1,493.4
1,TR-OY4HUB,A1,1,1,493.0
2,TR-OY4HUB,A1,2,1,504.2
3,TR-OY4HUB,A1,3,1,506.4
4,TR-OY4HUB,A1,4,1,514.7



Dtypes:
scenario_id        object
route_id           object
departure_slot      int64
leg                 int64
dmin_cm           float64
dtype: object

========== SCENARIO ==========
scenario_id: TR-OY4HUB
issue_time: 2015-01-30 06:00:00

========== WATER LEVELS ==========
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (504, 21)


,G01,G02,G03,G04,G05,G06,G07,G08,G09,G10,...,G12,G13,G14,G15,G16,G17,G18,G19,G20,G21
timestamp,,,,,,,,,,,,,,,,,,,,,
2015-01-09 06:00:00,1079.800049,170.500000,303.500000,325.299988,608.799988,527.200012,400.000000,1233.800049,366.799988,336.000000,...,714.200012,282.500000,513.000000,209.800003,449.000000,364.200012,648.000000,205.800003,680.500000,1272.000000
2015-01-09 07:00:00,1079.800049,169.500000,305.200012,325.500000,608.000000,525.799988,400.000000,1233.300049,369.500000,335.799988,...,713.799988,283.799988,512.500000,210.000000,449.500000,365.000000,647.500000,205.800003,679.799988,1272.300049
2015-01-09 08:00:00,1079.300049,169.199997,304.799988,327.200012,606.200012,525.000000,399.799988,1233.300049,374.500000,335.500000,...,713.500000,284.000000,512.000000,210.500000,449.799988,366.000000,645.500000,206.199997,679.799988,1272.000000
2015-01-09 09:00:00,1079.000000,168.000000,304.000000,328.200012,605.200012,524.799988,398.799988,1233.000000,374.500000,334.500000,...,713.000000,283.799988,512.000000,210.500000,449.799988,366.500000,645.000000,205.500000,679.799988,1271.500000
2015-01-09 10:00:00,1079.500000,168.000000,304.000000,329.200012,604.799988,525.000000,398.500000,1232.000000,371.500000,334.000000,...,712.200012,284.500000,512.200012,210.199997,450.799988,366.000000,644.200012,205.199997,680.799988,1271.199951



Columns/index:
['G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09', 'G10', 'G11', 'G12', 'G13', 'G14', 'G15', 'G16', 'G17', 'G18', 'G19', 'G20', 'G21']

========== DISCHARGE ==========
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (504, 16)


,Q01,Q02,Q03,Q04,Q05,Q06,Q07,Q08,Q09,Q10,Q11,Q12,Q13,Q14,Q15,Q16
timestamp,,,,,,,,,,,,,,,,
2015-01-09 06:00:00,4102.5,3980.0,4090.0,1400.0,1787.5,3840.0,1320.0,3220.0,1292.5,3372.5,3881.600098,3172.5,2297.5,3567.5,1253.199951,2180.0
2015-01-09 07:00:00,4100.0,3975.0,4090.0,1397.5,1787.5,3850.0,1317.5,3225.0,1305.0,3357.5,3887.600098,3180.0,2295.0,3560.0,1250.699951,2175.0
2015-01-09 08:00:00,4100.0,3955.0,4090.0,1395.0,1792.5,3835.0,1317.5,3227.5,1332.5,3350.0,3886.000000,3190.0,2292.5,3542.5,1250.099976,2170.0
2015-01-09 09:00:00,4100.0,3950.0,4085.0,1390.0,1785.0,3815.0,1317.5,3227.5,1332.5,3347.5,3872.600098,3195.0,2282.5,3532.5,1243.900024,2170.0
2015-01-09 10:00:00,4092.5,3950.0,4085.0,1387.5,1782.5,3805.0,1322.5,3237.5,1315.0,3350.0,3866.600098,3190.0,2280.0,3527.5,1233.800049,2172.5



Columns/index:
['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16']


In [37]:
import numpy as np
import pandas as pd

def make_scenario_features(data, scenarios):
    rows = []

    for _, sc in scenarios.iterrows():

        sid = sc["scenario_id"]
        issue = sc["issue_time"]

        # Historical data available at decision time
        levels = data.levels(sid, issue)
        discharge = data.discharge(sid, issue)

        # ---- Water level features ----
        level_features = {}

        for col in levels.columns:
            s = levels[col]

            level_features[f"{col}_last"] = s.iloc[-1]
            level_features[f"{col}_mean24"] = s.iloc[-24:].mean()
            level_features[f"{col}_min24"] = s.iloc[-24:].min()
            level_features[f"{col}_max24"] = s.iloc[-24:].max()
            level_features[f"{col}_change6"] = s.iloc[-1] - s.iloc[-7]
            level_features[f"{col}_change24"] = s.iloc[-1] - s.iloc[-25]
            level_features[f"{col}_mean7d"] = s.iloc[-168:].mean()

        # ---- Discharge features ----
        discharge_features = {}

        for col in discharge.columns:
            s = discharge[col]

            discharge_features[f"{col}_last"] = s.iloc[-1]
            discharge_features[f"{col}_mean24"] = s.iloc[-24:].mean()
            discharge_features[f"{col}_min24"] = s.iloc[-24:].min()
            discharge_features[f"{col}_max24"] = s.iloc[-24:].max()
            discharge_features[f"{col}_change24"] = s.iloc[-1] - s.iloc[-25]
            discharge_features[f"{col}_mean7d"] = s.iloc[-168:].mean()

        row = {
            "scenario_id": sid,
            "issue_time": issue,
            **level_features,
            **discharge_features
        }

        rows.append(row)

    return pd.DataFrame(rows)


scenario_features = make_scenario_features(
    data_train,
    train.scenarios
)

print("Feature dataset shape:", scenario_features.shape)
display(scenario_features.head())

Feature dataset shape: (596, 245)


,scenario_id,issue_time,G01_last,G01_mean24,G01_min24,G01_max24,G01_change6,G01_change24,G01_mean7d,G02_last,...,Q15_min24,Q15_max24,Q15_change24,Q15_mean7d,Q16_last,Q16_mean24,Q16_min24,Q16_max24,Q16_change24,Q16_mean7d
0,TR-OY4HUB,2015-01-30 06:00:00,945.700012,939.504150,932.700012,946.500000,3.400024,13.900024,961.729797,114.199997,...,846.200012,879.400024,39.800049,897.172058,1837.5,1813.541626,1765.0,1847.5,27.5,1860.892822
1,TR-LMQZDM,2015-01-30 06:00:00,945.700012,939.504150,932.700012,946.500000,3.400024,13.900024,961.729797,114.199997,...,846.200012,879.400024,39.800049,897.172058,1837.5,1813.541626,1765.0,1847.5,27.5,1860.892822
2,TR-MC096A,2015-02-04 06:00:00,1018.799988,1026.504150,1018.799988,1031.699951,-5.500061,-10.500061,978.031555,115.199997,...,1043.199951,1103.599976,-57.800049,970.136902,1802.5,1903.541626,1802.5,1962.5,-170.0,1961.577393
3,TR-LKYT1T,2015-02-04 06:00:00,1018.799988,1026.504150,1018.799988,1031.699951,-5.500061,-10.500061,978.031555,115.199997,...,1043.199951,1103.599976,-57.800049,970.136902,1802.5,1903.541626,1802.5,1962.5,-170.0,1961.577393
4,TR-NJ2T7X,2015-02-11 06:00:00,867.500000,872.200012,867.000000,878.700012,-1.799988,-11.500000,933.354187,38.200001,...,598.799988,634.500000,-30.299988,801.539795,1350.0,1360.937500,1330.0,1395.0,0.0,1545.967285


In [41]:
# ============================================
# BUILD THE ACTUAL ML TRAINING DATASET
# ============================================

# Keep only the scenario-level features
feature_cols = [
    c for c in scenario_features.columns
    if c not in ["scenario_id", "issue_time"]
]

# Merge scenario features into every target row
dataset = labels.merge(
    scenario_features,
    on="scenario_id",
    how="left"
)

print("Dataset shape:", dataset.shape)

print("\nMissing values:")
print(dataset.isna().sum().sum())

print("\nColumns:")
print(dataset.columns.tolist()[:20])

display(dataset.head())

Dataset shape: (59600, 249)

Missing values:
4600

Columns:
['scenario_id', 'route_id', 'departure_slot', 'leg', 'dmin_cm', 'issue_time', 'G01_last', 'G01_mean24', 'G01_min24', 'G01_max24', 'G01_change6', 'G01_change24', 'G01_mean7d', 'G02_last', 'G02_mean24', 'G02_min24', 'G02_max24', 'G02_change6', 'G02_change24', 'G02_mean7d']


,scenario_id,route_id,departure_slot,leg,dmin_cm,issue_time,G01_last,G01_mean24,G01_min24,G01_max24,...,Q15_min24,Q15_max24,Q15_change24,Q15_mean7d,Q16_last,Q16_mean24,Q16_min24,Q16_max24,Q16_change24,Q16_mean7d
0,TR-OY4HUB,A1,0,1,493.4,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,846.200012,879.400024,39.800049,897.172058,1837.5,1813.541626,1765.0,1847.5,27.5,1860.892822
1,TR-OY4HUB,A1,1,1,493.0,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,846.200012,879.400024,39.800049,897.172058,1837.5,1813.541626,1765.0,1847.5,27.5,1860.892822
2,TR-OY4HUB,A1,2,1,504.2,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,846.200012,879.400024,39.800049,897.172058,1837.5,1813.541626,1765.0,1847.5,27.5,1860.892822
3,TR-OY4HUB,A1,3,1,506.4,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,846.200012,879.400024,39.800049,897.172058,1837.5,1813.541626,1765.0,1847.5,27.5,1860.892822
4,TR-OY4HUB,A1,4,1,514.7,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,846.200012,879.400024,39.800049,897.172058,1837.5,1813.541626,1765.0,1847.5,27.5,1860.892822


In [20]:
X = dataset[features]
y = dataset["dmin_cm"

_IncompleteInputError: incomplete input (1195297529.py, line 2)

In [42]:
# ============================================
# TIME-BASED TRAIN / VALIDATION SPLIT
# ============================================

# Sort scenarios chronologically
scenario_dates = (
    train.scenarios[["scenario_id", "issue_time"]]
    .drop_duplicates()
    .sort_values("issue_time")
)

# 80% oldest scenarios -> training
# 20% newest scenarios -> validation
split_idx = int(len(scenario_dates) * 0.80)

train_scenarios = set(
    scenario_dates.iloc[:split_idx]["scenario_id"]
)

val_scenarios = set(
    scenario_dates.iloc[split_idx:]["scenario_id"]
)

train_df = dataset[
    dataset["scenario_id"].isin(train_scenarios)
].copy()

val_df = dataset[
    dataset["scenario_id"].isin(val_scenarios)
].copy()

print("Training scenarios:", len(train_scenarios))
print("Validation scenarios:", len(val_scenarios))

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(val_df))

print("\nTraining period:")
print(
    scenario_dates[
        scenario_dates.scenario_id.isin(train_scenarios)
    ]["issue_time"].min(),
    "→",
    scenario_dates[
        scenario_dates.scenario_id.isin(train_scenarios)
    ]["issue_time"].max()
)

print("\nValidation period:")
print(
    scenario_dates[
        scenario_dates.scenario_id.isin(val_scenarios)
    ]["issue_time"].min(),
    "→",
    scenario_dates[
        scenario_dates.scenario_id.isin(val_scenarios)
    ]["issue_time"].max()
)

Training scenarios: 476
Validation scenarios: 120

Training rows: 47600
Validation rows: 12000

Training period:
2015-01-30 06:00:00 → 2020-08-11 06:00:00

Validation period:
2020-08-15 06:00:00 → 2021-12-13 06:00:00


In [43]:
# ============================================
# PREPARE X AND y
# ============================================

# Columns that are NOT model features
drop_cols = [
    "scenario_id",
    "issue_time",
    "dmin_cm"
]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["dmin_cm"]

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["dmin_cm"]

# Convert categorical columns to category
for col in ["route_id"]:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

X_train: (47600, 246)
y_train: (47600,)
X_val: (12000, 246)
y_val: (12000,)

Features:
['route_id', 'departure_slot', 'leg', 'G01_last', 'G01_mean24', 'G01_min24', 'G01_max24', 'G01_change6', 'G01_change24', 'G01_mean7d', 'G02_last', 'G02_mean24', 'G02_min24', 'G02_max24', 'G02_change6', 'G02_change24', 'G02_mean7d', 'G03_last', 'G03_mean24', 'G03_min24', 'G03_max24', 'G03_change6', 'G03_change24', 'G03_mean7d', 'G04_last', 'G04_mean24', 'G04_min24', 'G04_max24', 'G04_change6', 'G04_change24', 'G04_mean7d', 'G05_last', 'G05_mean24', 'G05_min24', 'G05_max24', 'G05_change6', 'G05_change24', 'G05_mean7d', 'G06_last', 'G06_mean24', 'G06_min24', 'G06_max24', 'G06_change6', 'G06_change24', 'G06_mean7d', 'G07_last', 'G07_mean24', 'G07_min24', 'G07_max24', 'G07_change6', 'G07_change24', 'G07_mean7d', 'G08_last', 'G08_mean24', 'G08_min24', 'G08_max24', 'G08_change6', 'G08_change24', 'G08_mean7d', 'G09_last', 'G09_mean24', 'G09_min24', 'G09_max24', 'G09_change6', 'G09_change24', 'G09_mean7d', 'G

In [44]:
# ============================================
# FIRST REAL MODEL
# ============================================

from lightgbm import LGBMRegressor

model = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[]
)

print("Model trained successfully!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024677 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 51136
[LightGBM] [Info] Number of data points in the train set: 47600, number of used features: 246
[LightGBM] [Info] Start training from score 357.853236
Model trained successfully!


In [46]:
# ============================================
# EVALUATE OUR FIRST MODEL
# ============================================

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

pred_val = model.predict(X_val)

mae = mean_absolute_error(y_val, pred_val)
rmse = np.sqrt(mean_squared_error(y_val, pred_val))

print("========== OUR FIRST MODEL ==========")
print(f"MAE  : {mae:.2f} cm")
print(f"RMSE : {rmse:.2f} cm")

print("\nActual vs Predicted:")
comparison = pd.DataFrame({
    "actual": y_val.iloc[:20].values,
    "predicted": pred_val[:20]
})

display(comparison)

========== OUR FIRST MODEL ==========
MAE  : 40.82 cm
RMSE : 60.32 cm

Actual vs Predicted:


,actual,predicted
0,300.9,309.374202
1,294.5,310.104343
2,298.9,310.253137
3,291.7,308.214752
4,300.0,309.372547
5,292.7,308.368783
6,296.0,308.274092
7,286.2,306.576300
8,295.4,306.477487
9,286.7,308.415771


In [47]:
# ============================================
# BASELINE PINBALL LOSS FOR OUR FIRST MODEL
# ============================================

def pinball_loss(y_true, y_pred, q):
    error = y_true - y_pred

    return np.mean(
        np.where(
            error >= 0,
            q * error,
            (1 - q) * (-error)
        )
    )


# Our current model is just one prediction.
# Treat it as a q50 prediction for this diagnostic.
for q in [0.05, 0.10, 0.25, 0.50]:
    loss = pinball_loss(y_val.values, pred_val, q)
    print(f"q={q:.2f}  Pinball loss: {loss:.2f}")

overall = np.mean([
    pinball_loss(y_val.values, pred_val, q)
    for q in [0.05, 0.10, 0.25, 0.50]
])

print(f"\nAverage pinball loss: {overall:.2f}")

q=0.05  Pinball loss: 20.93
q=0.10  Pinball loss: 20.87
q=0.25  Pinball loss: 20.70
q=0.50  Pinball loss: 20.41

Average pinball loss: 20.73


In [48]:
from lightgbm import LGBMRegressor

q05_model = LGBMRegressor(
    objective="quantile",
    alpha=0.05,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

q05_model.fit(
    X_train,
    y_train
)

pred_q05 = q05_model.predict(X_val)

print("q05 model trained!")
print("First 10 predictions:")
print(pred_q05[:10])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.130661 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51136
[LightGBM] [Info] Number of data points in the train set: 47600, number of used features: 246
[LightGBM] [Info] Start training from score 196.190002
q05 model trained!
First 10 predictions:
[285.20982434 285.20982434 285.23393326 285.23393326 285.12724617
 285.08193738 285.07221524 284.90774019 284.78960966 284.75258194]


In [51]:
quantiles = [0.10, 0.25, 0.50]

quantile_models = {}
quantile_predictions = {
    0.05: pred_q05
}

for q in quantiles:
    model_q = LGBMRegressor(
        objective="quantile",
        alpha=q,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model_q.fit(X_train, y_train)

    quantile_models[q] = model_q
    quantile_predictions[q] = model_q.predict(X_val)

    print(f"q={q:.2f} trained!")

print("\nAll 4 quantile models are ready.")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.088779 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51136
[LightGBM] [Info] Number of data points in the train set: 47600, number of used features: 246
[LightGBM] [Info] Start training from score 227.199997
q=0.10 trained!
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.094728 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51136
[LightGBM] [Info] Number of data points in the train set: 47600, number of used features: 246
[LightGBM] [Info] Start training from score 272.500000
q=0.25 trained!
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.095443 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51136
[LightGBM] [Info] Number of data points in the train set: 47600, number of used

In [53]:
# ============================================
# CHECK QUANTILE ORDER
# ============================================

pred_q10 = quantile_predictions[0.10]
pred_q25 = quantile_predictions[0.25]
pred_q50 = quantile_predictions[0.50]

cross_1 = np.sum(pred_q05 > pred_q10)
cross_2 = np.sum(pred_q10 > pred_q25)
cross_3 = np.sum(pred_q25 > pred_q50)

print("Quantile crossing:")
print(f"q05 > q10 : {cross_1}")
print(f"q10 > q25 : {cross_2}")
print(f"q25 > q50 : {cross_3}")

# Show first 10 predictions
quantile_table = pd.DataFrame({
    "q05": pred_q05[:10],
    "q10": pred_q10[:10],
    "q25": pred_q25[:10],
    "q50": pred_q50[:10]
})

display(quantile_table)

Quantile crossing:
q05 > q10 : 2428
q10 > q25 : 1711
q25 > q50 : 3068


,q05,q10,q25,q50
0,285.209824,292.345898,292.994236,296.735054
1,285.209824,292.345898,292.736837,297.875473
2,285.233933,292.345898,292.767455,298.163899
3,285.233933,292.186532,291.578792,298.759222
4,285.127246,292.102092,291.793135,298.561482
5,285.081937,291.760353,292.761703,297.993930
6,285.072215,291.377890,292.795158,297.958031
7,284.907740,291.066632,291.323560,298.090694
8,284.789610,290.716624,291.249760,300.010074
9,284.752582,290.664618,290.681707,299.381975


In [54]:
# ============================================
# REAL 4-QUANTILE PINBALL LOSS
# ============================================

quantiles = [0.05, 0.10, 0.25, 0.50]

predictions = {
    0.05: pred_q05,
    0.10: pred_q10,
    0.25: pred_q25,
    0.50: pred_q50
}

losses = {}

for q in quantiles:
    losses[q] = pinball_loss(
        y_val.values,
        predictions[q],
        q
    )
    print(f"q={q:.2f}  Pinball loss: {losses[q]:.2f}")

overall_pinball = np.mean(list(losses.values()))

print("\n================================")
print(f"OUR MODEL SCORE: {overall_pinball:.2f}")
print("================================")

q=0.05  Pinball loss: 5.82
q=0.10  Pinball loss: 8.64
q=0.25  Pinball loss: 14.16
q=0.50  Pinball loss: 18.15

OUR MODEL SCORE: 11.69


In [ ]:
for name in ["pred_q05", "pred_q10", "pred_q25", "pred_q50"]:
    print(name, "exists:", name in globals())

In [55]:
# Put predictions into columns in quantile order
pred_matrix = np.column_stack([
    pred_q05,
    pred_q10,
    pred_q25,
    pred_q50
])

# Enforce q05 <= q10 <= q25 <= q50
pred_fixed = np.sort(pred_matrix, axis=1)

pred_q05_fixed = pred_fixed[:, 0]
pred_q10_fixed = pred_fixed[:, 1]
pred_q25_fixed = pred_fixed[:, 2]
pred_q50_fixed = pred_fixed[:, 3]

# Verify
print(
    "Crossings remaining:",
    np.sum(pred_fixed[:, 0] > pred_fixed[:, 1])
    + np.sum(pred_fixed[:, 1] > pred_fixed[:, 2])
    + np.sum(pred_fixed[:, 2] > pred_fixed[:, 3])
)

Crossings remaining: 0


In [56]:
# ============================================
# COMPARE SCORE BEFORE AND AFTER FIXING CROSSING
# ============================================

quantiles = [0.05, 0.10, 0.25, 0.50]

original_preds = [pred_q05, pred_q10, pred_q25, pred_q50]
fixed_preds = [
    pred_q05_fixed,
    pred_q10_fixed,
    pred_q25_fixed,
    pred_q50_fixed
]

print("Original model:")
original_losses = []

for q, pred in zip(quantiles, original_preds):
    loss = pinball_loss(y_val.values, pred, q)
    original_losses.append(loss)
    print(f"q={q:.2f}: {loss:.2f}")

print(f"Original average: {np.mean(original_losses):.2f}")

print("\nAfter fixing crossings:")
fixed_losses = []

for q, pred in zip(quantiles, fixed_preds):
    loss = pinball_loss(y_val.values, pred, q)
    fixed_losses.append(loss)
    print(f"q={q:.2f}: {loss:.2f}")

print(f"Fixed average: {np.mean(fixed_losses):.2f}")

Original model:
q=0.05: 5.82
q=0.10: 8.64
q=0.25: 14.16
q=0.50: 18.15
Original average: 11.69

After fixing crossings:
q=0.05: 5.55
q=0.10: 8.50
q=0.25: 13.80
q=0.50: 18.26
Fixed average: 11.53


In [57]:
# Inspect the route windows for one training scenario
sid = train.scenarios.iloc[0]["scenario_id"]

windows = data_train.windows
scenario_windows = windows[windows["scenario_id"] == sid]

print("Columns:")
print(scenario_windows.columns.tolist())

print("\nExample route windows:")
display(
    scenario_windows[
        ["route_id", "departure_slot", "leg", "segment_id",
         "t_enter", "t_exit", "equiv_depth_cm"]
    ].head(15)
)

Columns:
['route_id', 'departure_slot', 'leg', 'order', 'kind', 'segment_id', 'equiv_depth_cm', 't_enter', 't_exit', 'lock_h', 'scenario_id']

Example route windows:


,route_id,departure_slot,leg,segment_id,t_enter,t_exit,equiv_depth_cm
0,A1,0,1,None,2015-02-01 06:00:00.000000000,2015-02-01 08:59:30.229007633,470.0
1,A1,0,1,D02b,2015-02-01 08:59:30.229007633,2015-02-01 11:41:54.415054144,NaN
2,A1,0,1,D02a,2015-02-01 11:41:54.415054144,2015-02-01 14:27:47.903426237,NaN
3,A1,0,1,D01,2015-02-01 14:27:47.903426237,2015-02-01 15:04:37.205751818,NaN
4,A1,0,1,S17,2015-02-01 15:04:37.205751818,2015-02-01 16:19:37.205751818,NaN
5,A1,0,1,S16,2015-02-01 16:19:37.205751818,2015-02-01 19:00:52.205751818,NaN
6,A1,0,1,S15,2015-02-01 19:00:52.205751818,2015-02-01 19:50:22.205751818,NaN
7,A1,1,1,None,2015-02-01 12:00:00.000000000,2015-02-01 14:59:30.229007633,470.0
8,A1,1,1,D02b,2015-02-01 14:59:30.229007633,2015-02-01 17:41:54.415054144,NaN
9,A1,1,1,D02a,2015-02-01 17:41:54.415054144,2015-02-01 20:27:47.903426237,NaN


In [58]:
segments = pd.read_csv(f"{DATA}/reference/segments.csv")

print("Shape:", segments.shape)
print("Columns:", segments.columns.tolist())

display(segments.head(10))

Shape: (23, 11)
Columns: ['segment_id', 'country', 'km_from', 'km_to', 'length_km', 'segment_type', 'gauge_id', 'ref_standard', 'ref_status', 'ref_level_shifted_cm', 'guaranteed_depth_cm']


,segment_id,country,km_from,km_to,length_km,segment_type,gauge_id,ref_standard,ref_status,ref_level_shifted_cm,guaranteed_depth_cm
0,S01,DE,352.07,384.0,31.93,RIVER,G20,GlW2022,published,520.0,210
1,S02,DE,384.00,410.5,26.50,RIVER,G10,GlW2022,published,163.0,210
2,S03,DE,410.50,431.5,21.00,RIVER,G03,GlW2022,published,90.0,210
3,S04,DE,431.50,462.0,30.50,RIVER,G19,GlW2022,published,-2.0,210
4,S05,DE,462.00,508.0,46.00,RIVER,G14,GlW2022,published,311.0,210
5,S06,DE,508.00,540.0,32.00,RIVER,G15,GlW2022,published,47.0,190
6,S07,DE,540.00,557.0,17.00,RIVER,G02,GlW2022,published,-73.0,190
7,S08,DE,557.00,566.0,9.00,RIVER,G02,GlW2022,published,-73.0,210
8,S09,DE,566.00,592.2,26.20,RIVER,G13,GlW2022,published,12.0,210
9,S10,DE,592.20,601.0,8.80,RIVER,G13,GlW2022,published,12.0,250


In [59]:
print(segments.columns.tolist())
display(segments.head(10).T)

['segment_id', 'country', 'km_from', 'km_to', 'length_km', 'segment_type', 'gauge_id', 'ref_standard', 'ref_status', 'ref_level_shifted_cm', 'guaranteed_depth_cm']


,0,1,2,3,4,5,6,7,8,9
segment_id,S01,S02,S03,S04,S05,S06,S07,S08,S09,S10
country,DE,DE,DE,DE,DE,DE,DE,DE,DE,DE
km_from,352.07,384.0,410.5,431.5,462.0,508.0,540.0,557.0,566.0,592.2
km_to,384.0,410.5,431.5,462.0,508.0,540.0,557.0,566.0,592.2,601.0
length_km,31.93,26.5,21.0,30.5,46.0,32.0,17.0,9.0,26.2,8.8
segment_type,RIVER,RIVER,RIVER,RIVER,RIVER,RIVER,RIVER,RIVER,RIVER,RIVER
gauge_id,G20,G10,G03,G19,G14,G15,G02,G02,G13,G13
ref_standard,GlW2022,GlW2022,GlW2022,GlW2022,GlW2022,GlW2022,GlW2022,GlW2022,GlW2022,GlW2022
ref_status,published,published,published,published,published,published,published,published,published,published
ref_level_shifted_cm,520.0,163.0,90.0,-2.0,311.0,47.0,-73.0,-73.0,12.0,12.0


In [60]:
gauges = pd.read_csv(f"{DATA}/reference/gauges.csv")

print("Shape:", gauges.shape)
print("Columns:", gauges.columns.tolist())
display(gauges.head(10))

Shape: (21, 3)
Columns: ['gauge_id', 'rhine_km', 'country']


,gauge_id,rhine_km,country
0,G01,884.6,NL
1,G02,546.2,DE
2,G03,424.7,DE
3,G04,934.0,NL
4,G05,744.2,DE
5,G06,688.0,DE
6,G07,851.9,DE
7,G08,867.2,NL
8,G09,336.2,DE
9,G10,400.6,DE


In [63]:
segments = pd.read_csv(f"{DATA}/reference/segments.csv")

# Keep only river segments, matching the official function
segments = segments[
    segments["segment_type"] == "RIVER"
].copy()

print("River segments:", len(segments))


def make_segment_features(data, scenarios, segments):
    rows = []

    for _, sc in scenarios.iterrows():
        sid = sc["scenario_id"]
        issue = sc["issue_time"]

        levels = data.levels(sid, issue)
        latest_levels = levels.iloc[-1]

        row = {"scenario_id": sid}

        for _, seg in segments.iterrows():
            gauge = seg["gauge_id"]
            segment = seg["segment_id"]

            if pd.notna(gauge) and gauge in latest_levels.index:
                level = latest_levels[gauge]

                if pd.notna(level):
                    depth = (
                        seg["guaranteed_depth_cm"]
                        + level
                        - seg["ref_level_shifted_cm"]
                    )
                    row[f"{segment}_current_depth"] = depth

        rows.append(row)

    return pd.DataFrame(rows)


segment_features = make_segment_features(
    data_train,
    train.scenarios,
    segments
)

print("Feature shape:", segment_features.shape)
display(segment_features.head())

River segments: 22
Feature shape: (596, 23)


,scenario_id,S01_current_depth,S02_current_depth,S03_current_depth,S04_current_depth,S05_current_depth,S06_current_depth,S07_current_depth,S08_current_depth,S09_current_depth,...,S13_current_depth,S14_current_depth,S15_current_depth,S16_current_depth,S17_current_depth,D01_current_depth,D02a_current_depth,D02b_current_depth,D03_current_depth,D04_current_depth
0,TR-OY4HUB,352.799988,340.000000,374.200012,367.799988,375.799988,321.799988,377.200012,397.200012,447.000000,...,529.200012,530.799988,568.799988,584.0,558.5,566.5,565.699951,469.400024,562.699951,562.699951
1,TR-LMQZDM,352.799988,340.000000,374.200012,367.799988,375.799988,321.799988,377.200012,397.200012,447.000000,...,529.200012,530.799988,568.799988,584.0,558.5,566.5,565.699951,469.400024,562.699951,562.699951
2,TR-MC096A,333.000000,338.799988,367.200012,364.500000,371.200012,319.500000,378.200012,398.200012,446.000000,...,550.000000,564.000000,611.200012,642.0,627.0,641.0,638.800049,522.500000,635.000000,635.000000
3,TR-LKYT1T,333.000000,338.799988,367.200012,364.500000,371.200012,319.500000,378.200012,398.200012,446.000000,...,550.000000,564.000000,611.200012,642.0,627.0,641.0,638.800049,522.500000,635.000000,635.000000
4,TR-NJ2T7X,276.000000,281.500000,303.000000,297.500000,312.500000,271.000000,301.200012,321.200012,340.200012,...,414.799988,426.000000,466.000000,479.5,467.5,475.0,487.500000,401.900024,474.500000,474.500000


In [62]:
import inspect

print(inspect.signature(data_train.depth_from_levels))
print(inspect.getsource(data_train.depth_from_levels))

(lev: pandas.core.frame.DataFrame) -> pandas.core.frame.DataFrame
    def depth_from_levels(self, lev: pd.DataFrame) -> pd.DataFrame:
        """available depth per RIVER segment = guaranteed + (level - reference level).
        Both are shifted by the same undisclosed datum constant, so it cancels."""
        out = {}
        for _, s in self.segments[self.segments.segment_type == "RIVER"].iterrows():
            out[s.segment_id] = s.guaranteed_depth_cm + (lev[s.gauge_id] - s.ref_level_shifted_cm)
        return pd.DataFrame(out, index=lev.index)



In [64]:
# Add segment features to the existing labeled dataset
dataset_v2 = dataset.merge(
    segment_features,
    on="scenario_id",
    how="left"
)

print("Original dataset:", dataset.shape)
print("New dataset:", dataset_v2.shape)

# Check for missing feature values
new_cols = [
    c for c in segment_features.columns
    if c != "scenario_id"
]

print("Number of segment features:", len(new_cols))
print(
    "Missing segment feature values:",
    dataset_v2[new_cols].isna().sum().sum()
)

display(dataset_v2.head())

Original dataset: (59600, 249)
New dataset: (59600, 271)
Number of segment features: 22
Missing segment feature values: 0


,scenario_id,route_id,departure_slot,leg,dmin_cm,issue_time,G01_last,G01_mean24,G01_min24,G01_max24,...,S13_current_depth,S14_current_depth,S15_current_depth,S16_current_depth,S17_current_depth,D01_current_depth,D02a_current_depth,D02b_current_depth,D03_current_depth,D04_current_depth
0,TR-OY4HUB,A1,0,1,493.4,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,529.200012,530.799988,568.799988,584.0,558.5,566.5,565.699951,469.400024,562.699951,562.699951
1,TR-OY4HUB,A1,1,1,493.0,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,529.200012,530.799988,568.799988,584.0,558.5,566.5,565.699951,469.400024,562.699951,562.699951
2,TR-OY4HUB,A1,2,1,504.2,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,529.200012,530.799988,568.799988,584.0,558.5,566.5,565.699951,469.400024,562.699951,562.699951
3,TR-OY4HUB,A1,3,1,506.4,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,529.200012,530.799988,568.799988,584.0,558.5,566.5,565.699951,469.400024,562.699951,562.699951
4,TR-OY4HUB,A1,4,1,514.7,2015-01-30 06:00:00,945.700012,939.50415,932.700012,946.5,...,529.200012,530.799988,568.799988,584.0,558.5,566.5,565.699951,469.400024,562.699951,562.699951


In [70]:
# Sort scenarios chronologically
scenario_times = (
    dataset_v2[["scenario_id", "issue_time"]]
    .drop_duplicates()
    .sort_values("issue_time")
)

cutoff = int(len(scenario_times) * 0.8)

train_sids = set(
    scenario_times.iloc[:cutoff]["scenario_id"]
)
val_sids = set(
    scenario_times.iloc[cutoff:]["scenario_id"]
)

train_df_v2 = dataset_v2[
    dataset_v2["scenario_id"].isin(train_sids)
].copy()

val_df_v2 = dataset_v2[
    dataset_v2["scenario_id"].isin(val_sids)
].copy()

drop_cols = ["scenario_id", "issue_time", "dmin_cm"]

X_train_v2 = train_df_v2.drop(columns=drop_cols)
y_train_v2 = train_df_v2["dmin_cm"]

X_val_v2 = val_df_v2.drop(columns=drop_cols)
y_val_v2 = val_df_v2["dmin_cm"]

# Tell LightGBM that route_id is categorical
for X in [X_train_v2, X_val_v2]:
    X["route_id"] = X["route_id"].astype("category")

print("Training shape:", X_train_v2.shape)
print("Validation shape:", X_val_v2.shape)
print("Training scenarios:", len(train_sids))
print("Validation scenarios:", len(val_sids))

Training shape: (47600, 268)
Validation shape: (12000, 268)
Training scenarios: 476
Validation scenarios: 120


In [71]:
# Sort scenarios chronologically
scenario_times = (
    dataset_v2[["scenario_id", "issue_time"]]
    .drop_duplicates()
    .sort_values("issue_time")
)

cutoff = int(len(scenario_times) * 0.8)

train_sids = set(
    scenario_times.iloc[:cutoff]["scenario_id"]
)
val_sids = set(
    scenario_times.iloc[cutoff:]["scenario_id"]
)

train_df_v2 = dataset_v2[
    dataset_v2["scenario_id"].isin(train_sids)
].copy()

val_df_v2 = dataset_v2[
    dataset_v2["scenario_id"].isin(val_sids)
].copy()

drop_cols = ["scenario_id", "issue_time", "dmin_cm"]

X_train_v2 = train_df_v2.drop(columns=drop_cols)
y_train_v2 = train_df_v2["dmin_cm"]

X_val_v2 = val_df_v2.drop(columns=drop_cols)
y_val_v2 = val_df_v2["dmin_cm"]

# Tell LightGBM that route_id is categorical
for X in [X_train_v2, X_val_v2]:
    X["route_id"] = X["route_id"].astype("category")

print("Training shape:", X_train_v2.shape)
print("Validation shape:", X_val_v2.shape)
print("Training scenarios:", len(train_sids))
print("Validation scenarios:", len(val_sids))

Training shape: (47600, 268)
Validation shape: (12000, 268)
Training scenarios: 476
Validation scenarios: 120


In [72]:
pred_matrix_v2 = np.column_stack([
    quantile_predictions_v2[q] for q in quantiles
])

# Ensure predictions are in ascending quantile order
pred_fixed_v2 = np.sort(pred_matrix_v2, axis=1)

losses_fixed_v2 = {
    q: pinball_loss(y_val_v2, pred_fixed_v2[:, i], q)
    for i, q in enumerate(quantiles)
}

score_v2 = np.mean(list(losses_fixed_v2.values()))

print("Pinball losses after sorting:")
for q in quantiles:
    print(f"q={q:.2f}: {losses_fixed_v2[q]:.4f}")

print(f"\nNew average validation score: {score_v2:.4f}")
print("Previous average validation score: 11.53")
print(f"Improvement: {11.53 - score_v2:.4f}")

NameError: name 'quantile_predictions_v2' is not defined

In [73]:
print("X_train_v2 exists:", "X_train_v2" in globals())
print("y_train_v2 exists:", "y_train_v2" in globals())
print("X_val_v2 exists:", "X_val_v2" in globals())
print("quantile_predictions_v2 exists:", "quantile_predictions_v2" in globals())

X_train_v2 exists: True
y_train_v2 exists: True
X_val_v2 exists: True
quantile_predictions_v2 exists: False


In [74]:
from lightgbm import LGBMRegressor
import numpy as np

quantiles = [0.05, 0.10, 0.25, 0.50]
quantile_models_v2 = {}
quantile_predictions_v2 = {}

for q in quantiles:
    print(f"Training q={q}...", flush=True)

    model = LGBMRegressor(
        objective="quantile",
        alpha=q,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(X_train_v2, y_train_v2)

    quantile_models_v2[q] = model
    quantile_predictions_v2[q] = model.predict(X_val_v2)

print("\nAll four models trained successfully.")
print("Prediction lengths:", {
    q: len(quantile_predictions_v2[q]) for q in quantiles
})

Training q=0.05...
Training q=0.1...
Training q=0.25...
Training q=0.5...

All four models trained successfully.
Prediction lengths: {0.05: 12000, 0.1: 12000, 0.25: 12000, 0.5: 12000}


In [75]:
pred_matrix_v2 = np.column_stack([
    quantile_predictions_v2[q] for q in quantiles
])

pred_fixed_v2 = np.sort(pred_matrix_v2, axis=1)

losses_fixed_v2 = {
    q: pinball_loss(y_val_v2, pred_fixed_v2[:, i], q)
    for i, q in enumerate(quantiles)
}

score_v2 = np.mean(list(losses_fixed_v2.values()))

for q, loss in losses_fixed_v2.items():
    print(f"q={q:.2f}: {loss:.4f}")

print(f"\nNew average validation score: {score_v2:.4f}")
print("Previous score: 11.53")

q=0.05: 5.4062
q=0.10: 8.4517
q=0.25: 14.2246
q=0.50: 18.6939

New average validation score: 11.6941
Previous score: 11.53


In [76]:
print("Original q05 model:", "q05_model" in globals())
print("Original quantile models:", "quantile_models" in globals())
print("Original q05 predictions:", "pred_q05" in globals())
print("Original q10 predictions:", "pred_q10" in globals())
print("Original q25 predictions:", "pred_q25" in globals())
print("Original q50 predictions:", "pred_q50" in globals())

Original q05 model: True
Original quantile models: True
Original q05 predictions: True
Original q10 predictions: True
Original q25 predictions: True
Original q50 predictions: True


In [77]:


import numpy as np

quantiles = [0.05, 0.10, 0.25, 0.50]

pred_matrix = np.column_stack([
    pred_q05,
    pred_q10,
    pred_q25,
    pred_q50
])

# Enforce q05 <= q10 <= q25 <= q50
pred_fixed = np.sort(pred_matrix, axis=1)

pred_q05_fixed = pred_fixed[:, 0]
pred_q10_fixed = pred_fixed[:, 1]
pred_q25_fixed = pred_fixed[:, 2]
pred_q50_fixed = pred_fixed[:, 3]

print("Number of validation predictions:", len(pred_q05_fixed))
print("Quantile crossings remaining:",
      np.sum(np.diff(pred_fixed, axis=1) < 0))
print("First 5 corrected prediction rows:")
print(pred_fixed[:5])

Number of validation predictions: 12000
Quantile crossings remaining: 0
First 5 corrected prediction rows:
[[285.20982434 292.34589809 292.99423557 296.73505365]
 [285.20982434 292.34589809 292.73683703 297.87547337]
 [285.23393326 292.34589809 292.76745485 298.1638992 ]
 [285.23393326 291.5787923  292.18653167 298.75922165]
 [285.12724617 291.79313455 292.10209226 298.5614821 ]]


In [78]:
# Generate historical features for test scenarios
scenario_features_test = make_scenario_features(
    data_test if "data_test" in globals()
    else Data(DATA, "test", env=test),
    test.scenarios
)

print("Test scenario features:", scenario_features_test.shape)
display(scenario_features_test.head())

Test scenario features: (94, 245)


,scenario_id,issue_time,G01_last,G01_mean24,G01_min24,G01_max24,G01_change6,G01_change24,G01_mean7d,G02_last,...,Q15_min24,Q15_max24,Q15_change24,Q15_mean7d,Q16_last,Q16_mean24,Q16_min24,Q16_max24,Q16_change24,Q16_mean7d
0,TS-OOKXQ1,2022-02-01 06:00:00,820.799988,822.333313,820.299988,825.000000,-0.400024,1.099976,828.380981,-4.8,...,361.700012,369.600006,8.399994,404.650024,1100.0,1098.645874,1090.0,1100.0,0.0,1150.431519
1,TS-E3P9HO,2022-02-01 06:00:00,820.799988,822.333313,820.299988,825.000000,-0.400024,1.099976,828.380981,-4.8,...,361.700012,369.600006,8.399994,404.650024,1100.0,1098.645874,1090.0,1100.0,0.0,1150.431519
2,TS-35WBG9,2022-03-06 06:00:00,851.500000,858.537537,851.500000,865.799988,-2.799988,-14.700012,926.563599,21.0,...,551.200012,628.099976,-74.299988,784.168457,1220.0,1258.958374,1220.0,1307.5,-82.5,1456.413696
3,TS-OX9OC7,2022-03-06 06:00:00,851.500000,858.537537,851.500000,865.799988,-2.799988,-14.700012,926.563599,21.0,...,551.200012,628.099976,-74.299988,784.168457,1220.0,1258.958374,1220.0,1307.5,-82.5,1456.413696
4,TS-DUCKBM,2022-04-20 06:00:00,839.799988,846.366699,839.799988,852.000000,-5.500000,-12.400024,931.593445,22.0,...,482.700012,536.000000,-55.299988,779.547607,1220.0,1246.770874,1217.5,1280.0,-40.0,1549.330322


In [79]:
# Get the test prediction rows
test_rows = test.manifests.copy()

print("Test manifest columns:", test_rows.columns.tolist())
print("Test manifest shape:", test_rows.shape)

display(test_rows.head())

Test manifest columns: ['scenario_id', 'container_id', 'port', 'is_empty', 'mass_t', 'value_eur', 'deadline', 'late_penalty_eur_h']
Test manifest shape: (20326, 8)


,scenario_id,container_id,port,is_empty,mass_t,value_eur,deadline,late_penalty_eur_h
0,TS-OOKXQ1,C001,Duisburg,0,12.56,373.0,2022-02-06 22:00:00,9.4
1,TS-OOKXQ1,C002,Duisburg,0,12.41,275.0,2022-02-08 19:00:00,13.2
2,TS-OOKXQ1,C003,Duisburg,0,30.48,676.0,2022-02-09 01:00:00,11.4
3,TS-OOKXQ1,C004,Duisburg,1,2.20,50.0,2022-02-16 06:00:00,0.5
4,TS-OOKXQ1,C005,Duisburg,1,2.20,74.0,2022-02-16 06:00:00,0.5


In [80]:
print("Test manifest columns:", test.manifests.columns.tolist())
print("Test manifest shape:", test.manifests.shape)
display(test.manifests.head())

Test manifest columns: ['scenario_id', 'container_id', 'port', 'is_empty', 'mass_t', 'value_eur', 'deadline', 'late_penalty_eur_h']
Test manifest shape: (20326, 8)


,scenario_id,container_id,port,is_empty,mass_t,value_eur,deadline,late_penalty_eur_h
0,TS-OOKXQ1,C001,Duisburg,0,12.56,373.0,2022-02-06 22:00:00,9.4
1,TS-OOKXQ1,C002,Duisburg,0,12.41,275.0,2022-02-08 19:00:00,13.2
2,TS-OOKXQ1,C003,Duisburg,0,30.48,676.0,2022-02-09 01:00:00,11.4
3,TS-OOKXQ1,C004,Duisburg,1,2.20,50.0,2022-02-16 06:00:00,0.5
4,TS-OOKXQ1,C005,Duisburg,1,2.20,74.0,2022-02-16 06:00:00,0.5


In [81]:
print("Sample submission:")
print(sample.head(10))
print("\nSample columns:", sample.columns.tolist())
print("Sample shape:", sample.shape)

print("\nTest scenarios:")
print(test.scenarios.head())
print("Test scenario columns:", test.scenarios.columns.tolist())

Sample submission:
                 id  dmin_q05_cm  dmin_q10_cm  dmin_q25_cm  dmin_q50_cm
0  TS-OOKXQ1_A1_0_1        200.0        210.0        230.0        250.0
1  TS-OOKXQ1_A1_1_1        200.0        210.0        230.0        250.0
2  TS-OOKXQ1_A1_2_1        200.0        210.0        230.0        250.0
3  TS-OOKXQ1_A1_3_1        200.0        210.0        230.0        250.0
4  TS-OOKXQ1_A1_4_1        200.0        210.0        230.0        250.0
5  TS-OOKXQ1_A1_5_1        200.0        210.0        230.0        250.0
6  TS-OOKXQ1_A1_6_1        200.0        210.0        230.0        250.0
7  TS-OOKXQ1_A1_7_1        200.0        210.0        230.0        250.0
8  TS-OOKXQ1_A1_8_1        200.0        210.0        230.0        250.0
9  TS-OOKXQ1_A1_9_1        200.0        210.0        230.0        250.0

Sample columns: ['id', 'dmin_q05_cm', 'dmin_q10_cm', 'dmin_q25_cm', 'dmin_q50_cm']
Sample shape: (9400, 5)

Test scenarios:
  scenario_id  issue_id family     origin                       

In [82]:
test_rows = sample[["id"]].copy()

parts = test_rows["id"].str.rsplit("_", n=3, expand=True)

test_rows["scenario_id"] = parts[0]
test_rows["route_id"] = parts[1]
test_rows["departure_slot"] = parts[2].astype(int)
test_rows["leg"] = parts[3].astype(int)

print("Prediction rows:", len(test_rows))
print("Unique scenarios:", test_rows["scenario_id"].nunique())
print("Routes:", sorted(test_rows["route_id"].unique()))
print("Departure slots:", sorted(test_rows["departure_slot"].unique()))
print("Legs:", sorted(test_rows["leg"].unique()))

display(test_rows.head())

Prediction rows: 9400
Unique scenarios: 94
Routes: ['A1', 'A2', 'B1', 'B2']
Departure slots: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
Legs: [np.int64(1), np.int64(2), np.int64(3)]


,id,scenario_id,route_id,departure_slot,leg
0,TS-OOKXQ1_A1_0_1,TS-OOKXQ1,A1,0,1
1,TS-OOKXQ1_A1_1_1,TS-OOKXQ1,A1,1,1
2,TS-OOKXQ1_A1_2_1,TS-OOKXQ1,A1,2,1
3,TS-OOKXQ1_A1_3_1,TS-OOKXQ1,A1,3,1
4,TS-OOKXQ1_A1_4_1,TS-OOKXQ1,A1,4,1


In [83]:
# Attach historical scenario features to each required prediction row
test_features = test_rows.merge(
    scenario_features_test,
    on="scenario_id",
    how="left",
    validate="many_to_one"
)

print("Test features shape:", test_features.shape)

# Check whether any historical features are missing
feature_cols = [
    c for c in scenario_features_test.columns
    if c not in ["scenario_id", "issue_time"]
]

print(
    "Missing historical feature values:",
    test_features[feature_cols].isna().sum().sum()
)

display(test_features.head())

Test features shape: (9400, 249)
Missing historical feature values: 1000


,id,scenario_id,route_id,departure_slot,leg,issue_time,G01_last,G01_mean24,G01_min24,G01_max24,...,Q15_min24,Q15_max24,Q15_change24,Q15_mean7d,Q16_last,Q16_mean24,Q16_min24,Q16_max24,Q16_change24,Q16_mean7d
0,TS-OOKXQ1_A1_0_1,TS-OOKXQ1,A1,0,1,2022-02-01 06:00:00,820.799988,822.333313,820.299988,825.0,...,361.700012,369.600006,8.399994,404.650024,1100.0,1098.645874,1090.0,1100.0,0.0,1150.431519
1,TS-OOKXQ1_A1_1_1,TS-OOKXQ1,A1,1,1,2022-02-01 06:00:00,820.799988,822.333313,820.299988,825.0,...,361.700012,369.600006,8.399994,404.650024,1100.0,1098.645874,1090.0,1100.0,0.0,1150.431519
2,TS-OOKXQ1_A1_2_1,TS-OOKXQ1,A1,2,1,2022-02-01 06:00:00,820.799988,822.333313,820.299988,825.0,...,361.700012,369.600006,8.399994,404.650024,1100.0,1098.645874,1090.0,1100.0,0.0,1150.431519
3,TS-OOKXQ1_A1_3_1,TS-OOKXQ1,A1,3,1,2022-02-01 06:00:00,820.799988,822.333313,820.299988,825.0,...,361.700012,369.600006,8.399994,404.650024,1100.0,1098.645874,1090.0,1100.0,0.0,1150.431519
4,TS-OOKXQ1_A1_4_1,TS-OOKXQ1,A1,4,1,2022-02-01 06:00:00,820.799988,822.333313,820.299988,825.0,...,361.700012,369.600006,8.399994,404.650024,1100.0,1098.645874,1090.0,1100.0,0.0,1150.431519


In [84]:
# Count missing values by feature
missing_by_feature = (
    test_features[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Features with missing values:")
display(missing_by_feature[missing_by_feature > 0])

# Count affected scenarios
missing_by_scenario = (
    test_features.groupby("scenario_id")[feature_cols]
    .apply(lambda df: df.isna().sum().sum())
)

print("\nScenarios with missing feature values:")
display(missing_by_scenario[missing_by_scenario > 0].head(20))

print("\nTotal affected scenarios:", (missing_by_scenario > 0).sum())

Features with missing values:


Q15_min24       200
Q15_mean24      200
Q15_max24       200
Q15_last        200
Q15_change24    200
dtype: int64


Scenarios with missing feature values:


scenario_id
TS-4DSJ2T    750
TS-KK36EF    250
dtype: int64


Total affected scenarios: 2


In [86]:
# Recreate the exact feature order used by the original model
model_features = list(X_train.columns)

missing_columns = [
    c for c in model_features
    if c not in test_features.columns
]

extra_columns = [
    c for c in test_features.columns
    if c not in model_features and c not in ["id", "scenario_id", "issue_time"]
]

print("Model feature count:", len(model_features))
print("Missing model columns:", missing_columns)
print("Unexpected extra columns:", extra_columns)

assert not missing_columns, "Test data is missing model features!"

Model feature count: 246
Missing model columns: []
Unexpected extra columns: []


In [87]:
# Select model features in the exact training order
X_test_final = test_features[model_features].copy()

# Match categorical type used during training
X_test_final["route_id"] = X_test_final["route_id"].astype("category")

# Predict each quantile using the original models
test_predictions = np.column_stack([
    quantile_models[0.05].predict(X_test_final),
    quantile_models[0.10].predict(X_test_final),
    quantile_models[0.25].predict(X_test_final),
    quantile_models[0.50].predict(X_test_final),
])

# Enforce ascending quantiles for every row
test_predictions = np.sort(test_predictions, axis=1)

print("Prediction shape:", test_predictions.shape)
print("Missing predictions:", np.isnan(test_predictions).sum())
print("Minimum prediction:", test_predictions.min())
print("Maximum prediction:", test_predictions.max())
print("Quantile crossings remaining:",
      np.sum(np.diff(test_predictions, axis=1) < 0))

KeyError: 0.05

In [88]:
print("quantile_models keys:", list(quantile_models.keys()))

print("quantile_models_v2 keys:",
      list(quantile_models_v2.keys())
      if "quantile_models_v2" in globals() else "Not found")

print("Separate q05 model exists:", "q05_model" in globals())

quantile_models keys: [0.1, 0.25, 0.5]
quantile_models_v2 keys: [0.05, 0.1, 0.25, 0.5]
Separate q05 model exists: True


In [89]:
import numpy as np

# Use the original models, not v2
test_predictions = np.column_stack([
    q05_model.predict(X_test_final),
    quantile_models[0.1].predict(X_test_final),
    quantile_models[0.25].predict(X_test_final),
    quantile_models[0.5].predict(X_test_final),
])

# Ensure q05 <= q10 <= q25 <= q50
test_predictions = np.sort(test_predictions, axis=1)

print("Prediction shape:", test_predictions.shape)
print("Missing predictions:", np.isnan(test_predictions).sum())
print("Minimum prediction:", test_predictions.min())
print("Maximum prediction:", test_predictions.max())
print(
    "Quantile crossings remaining:",
    np.sum(np.diff(test_predictions, axis=1) < 0)
)

Prediction shape: (9400, 4)
Missing predictions: 0
Minimum prediction: 145.78443821376104
Maximum prediction: 728.9109301842024
Quantile crossings remaining: 0


In [91]:
import numpy as np

def pinball_loss(y_true, y_pred, q):
    error = y_true - y_pred
    return np.mean(
        np.where(
            error >= 0,
            q * error,
            (1 - q) * (-error)
        )
    )

# Use the predictions AFTER quantile sorting
validation_predictions = np.column_stack([
    pred_q05_fixed,
    pred_q10_fixed,
    pred_q25_fixed,
    pred_q50_fixed
])

quantiles = [0.05, 0.10, 0.25, 0.50]

losses = [
    pinball_loss(y_val.to_numpy(), validation_predictions[:, i], q)
    for i, q in enumerate(quantiles)
]

print("=== VALIDATION PERFORMANCE ===")
for q, loss in zip(quantiles, losses):
    print(f"q{int(q*100):02d} Pinball Loss: {loss:.4f}")

average_loss = np.mean(losses)
print(f"\nAverage Pinball Loss: {average_loss:.4f}")

=== VALIDATION PERFORMANCE ===
q05 Pinball Loss: 5.5521
q10 Pinball Loss: 8.4998
q25 Pinball Loss: 13.7961
q50 Pinball Loss: 18.2613

Average Pinball Loss: 11.5273


In [92]:
baseline_loss = 20.73  # Your earlier baseline result

improvement = (
    (baseline_loss - average_loss) / baseline_loss
) * 100

print(f"Improvement over baseline: {improvement:.2f}%")

Improvement over baseline: 44.39%


In [93]:
submission = sample.copy()

submission[
    ["dmin_q05_cm", "dmin_q10_cm", "dmin_q25_cm", "dmin_q50_cm"]
] = test_predictions

# Validate
assert len(submission) == len(sample) == 9400
assert submission["id"].equals(sample["id"])
assert list(submission.columns) == list(sample.columns)
assert submission.iloc[:, 1:].notna().all().all()
assert np.isfinite(submission.iloc[:, 1:].to_numpy()).all()
assert np.all(np.diff(submission.iloc[:, 1:].to_numpy(), axis=1) >= 0)

# Save locally in Kaggle
output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)

print("All checks passed!")
print("Rows:", len(submission))
print("Saved to:", output_path)

display(submission.head())

All checks passed!
Rows: 9400
Saved to: /kaggle/working/submission.csv


,id,dmin_q05_cm,dmin_q10_cm,dmin_q25_cm,dmin_q50_cm
0,TS-OOKXQ1_A1_0_1,350.266406,358.820584,374.338635,380.156841
1,TS-OOKXQ1_A1_1_1,350.266406,359.139881,374.338635,385.683801
2,TS-OOKXQ1_A1_2_1,351.208827,359.926612,374.767805,385.984973
3,TS-OOKXQ1_A1_3_1,351.960403,360.376979,375.535668,387.312967
4,TS-OOKXQ1_A1_4_1,352.026843,360.919728,375.920405,387.204324


In [94]:
import numpy as np

def pinball_loss(y_true, y_pred, q):
    error = y_true - y_pred
    return np.mean(
        np.where(error >= 0, q * error, (1 - q) * (-error))
    )

validation_predictions = np.column_stack([
    pred_q05_fixed,
    pred_q10_fixed,
    pred_q25_fixed,
    pred_q50_fixed
])

quantiles = [0.05, 0.10, 0.25, 0.50]

losses = [
    pinball_loss(y_val.to_numpy(), validation_predictions[:, i], q)
    for i, q in enumerate(quantiles)
]

for q, loss in zip(quantiles, losses):
    print(f"q{int(q * 100):02d} loss: {loss:.4f}")

print(f"\nAverage validation pinball loss: {np.mean(losses):.4f}")

q05 loss: 5.5521
q10 loss: 8.4998
q25 loss: 13.7961
q50 loss: 18.2613

Average validation pinball loss: 11.5273


In [95]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": quantile_models[0.5].feature_importances_
})

importance = importance.sort_values(
    "importance", ascending=False
)

display(importance.head(20))

,feature,importance
1,departure_slot,1476
2,leg,1291
0,route_id,618
3,G01_last,405
140,G20_change6,329
14,G02_change6,320
63,G09_change6,318
21,G03_change6,292
29,G04_change24,254
98,G14_change6,252


In [96]:
from lightgbm import LGBMRegressor
import numpy as np

q50_small = LGBMRegressor(
    objective="quantile",
    alpha=0.5,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=6,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

q50_small.fit(X_train, y_train)

pred_q50_small = q50_small.predict(X_val)

old_loss = pinball_loss(y_val.to_numpy(), pred_q50_fixed, 0.5)
new_loss = pinball_loss(y_val.to_numpy(), pred_q50_small, 0.5)

print(f"Original q50 loss: {old_loss:.4f}")
print(f"Smaller model q50 loss: {new_loss:.4f}")
print(f"Difference: {new_loss - old_loss:+.4f}")

Original q50 loss: 18.2613
Smaller model q50 loss: 18.7229
Difference: +0.4616


In [97]:
q50_more = LGBMRegressor(
    objective="quantile",
    alpha=0.5,
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

q50_more.fit(X_train, y_train)

pred_q50_more = q50_more.predict(X_val)

new_loss = pinball_loss(y_val.to_numpy(), pred_q50_more, 0.5)

print(f"Original q50 loss: {old_loss:.4f}")
print(f"More-trees q50 loss: {new_loss:.4f}")
print(f"Difference: {new_loss - old_loss:+.4f}")

Original q50 loss: 18.2613
More-trees q50 loss: 18.1926
Difference: -0.0687


In [98]:
from lightgbm import LGBMRegressor
import numpy as np

new_models = {}
new_predictions = {}

for q in [0.05, 0.10, 0.25]:
    model_new = LGBMRegressor(
        objective="quantile",
        alpha=q,
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model_new.fit(X_train, y_train)
    new_models[q] = model_new
    new_predictions[q] = model_new.predict(X_val)

    original_pred = {
        0.05: pred_q05_fixed,
        0.10: pred_q10_fixed,
        0.25: pred_q25_fixed
    }[q]

    print(
        f"q{int(q*100):02d} | "
        f"original={pinball_loss(y_val.to_numpy(), original_pred, q):.4f} | "
        f"new={pinball_loss(y_val.to_numpy(), new_predictions[q], q):.4f}"
    )

q05 | original=5.5521 | new=5.8298
q10 | original=8.4998 | new=8.5851
q25 | original=13.7961 | new=14.8278


import os
import pandas as pd

path = "/kaggle/working/submission.csv"

print("File exists:", os.path.exists(path))

if os.path.exists(path):
    sub = pd.read_csv(path)
    print("Shape:", sub.shape)
    print(sub.head())
    print(sub.isna().sum())

In [1]:
import os
import pandas as pd

path = "/kaggle/working/submission.csv"

print("File exists:", os.path.exists(path))

if os.path.exists(path):
    sub = pd.read_csv(path)
    print("Shape:", sub.shape)
    print(sub.head())
    print(sub.isna().sum())

File exists: False


In [14]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if "submission" in file.lower() or file.endswith(".csv"):
            print(os.path.join(root, file))